# LeetCode #11 — Container With Most Water: nhập môn Two Pointers hai đầu

**Mức độ:** Medium · **Chủ đề:** Array, Two Pointers, Greedy

Skeleton trên LeetCode:

```python
class Solution:
    def maxArea(self, height: List[int]) -> int:
```

Mục tiêu của notebook là giúp bạn hiểu **kỹ thuật hai con trỏ đi từ hai đầu vào giữa (opposite-direction two pointers)**, và quan trọng hơn: **lập luận vì sao được phép bỏ qua** rất nhiều cặp mà vẫn không mất đáp án.

---

## Sau notebook này, bạn sẽ nắm được

1. Cách tính diện tích "bể nước" giữa hai cột.
2. Vì sao thử mọi cặp tốn `O(n²)`.
3. Ý tưởng hai con trỏ `left`, `right` và **quy tắc dời cột thấp hơn**.
4. **Chứng minh** quy tắc đó không bỏ sót đáp án — phần quan trọng nhất của bài.
5. Lỗi thường gặp, mẹo nhận diện, bài tập từ dễ đến nâng cao.

# 1. Bài toán là gì?

Cho mảng `height` dài `n`. Có `n` cột thẳng đứng, cột `i` đứng ở hoành độ `i` và cao `height[i]`.

Chọn **hai cột** để cùng trục hoành tạo thành một cái bể. Hỏi bể chứa được **nhiều nước nhất** là bao nhiêu? (Không được nghiêng bể.)

**Constraints (theo đề):**

- `n == height.length`, `2 <= n <= 10^5`
- `0 <= height[i] <= 10^4`

---

## Ví dụ 1

```text
height = [1, 8, 6, 2, 5, 4, 8, 3, 7]  ->  49
```

Chọn cột ở chỉ số `1` (cao 8) và chỉ số `8` (cao 7): chiều rộng `8 - 1 = 7`, mực nước bị giới hạn bởi cột **thấp hơn** là `7`, diện tích `7 * 7 = 49`.

## Ví dụ 2

```text
height = [1, 1]  ->  1
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Hai con trỏ (two pointers)
2. Thuật toán tham lam (greedy)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Hai con trỏ (two pointers)

### Ý tưởng

Dùng **hai chỉ số** di chuyển trên mảng/chuỗi theo quy tắc, để thay vòng lặp lồng `O(n²)` bằng một lượt `O(n)`. Hai kiểu chính:

**1. Hai đầu tiến vào nhau** (thường trên mảng đã sắp xếp):

```text
[1, 2, 4, 7, 11, 15], tìm hai số có tổng 15
 L                 R     1 + 15 = 16 > 15 → R lùi (cần nhỏ hơn)
 L             R         1 + 11 = 12 < 15 → L tiến (cần lớn hơn)
    L          R         2 + 11 = 13 < 15 → L tiến
       L       R         4 + 11 = 15 ✓
```

Vì sao được phép bỏ? Khi `a[L] + a[R] > target`, mọi cặp `(L', R)` với `L' > L` còn lớn hơn → `R` không thể là đáp án, bỏ `R`.

**2. Cùng chiều — con trỏ đọc và con trỏ ghi** (sửa mảng tại chỗ):

```text
xóa mọi số 0, giữ thứ tự: [0, 1, 0, 3, 12]
read đi qua từng phần tử; gặp số khác 0 thì chép vào vị trí write, write += 1
→ [1, 3, 12, ...]
```

In [1]:
def tong_hai_so_da_sap_xep(a, target):
    L, R = 0, len(a) - 1
    while L < R:
        s = a[L] + a[R]
        print(f"   L={L} R={R}: {a[L]} + {a[R]} = {s}")
        if s == target:
            return L, R
        if s < target:
            L += 1
        else:
            R -= 1
    return None

print("→", tong_hai_so_da_sap_xep([1, 2, 4, 7, 11, 15], 15))

def bo_so_khong(a):
    write = 0
    for read in range(len(a)):
        if a[read] != 0:
            a[write] = a[read]
            write += 1
    return a[:write]

print("bỏ số 0:", bo_so_khong([0, 1, 0, 3, 12]))

def la_palindrome(s):
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return False
        i += 1; j -= 1
    return True
print("'racecar' palindrome?", la_palindrome("racecar"), "| 'abca'?", la_palindrome("abca"))

   L=0 R=5: 1 + 15 = 16
   L=0 R=4: 1 + 11 = 12
   L=1 R=4: 2 + 11 = 13
   L=2 R=4: 4 + 11 = 15
→ (2, 4)
bỏ số 0: [1, 3, 12]
'racecar' palindrome? True | 'abca'? False


### Đảo ngược một đoạn tại chỗ

Hai con trỏ ở hai đầu, đổi chỗ rồi tiến vào trong — dùng rất nhiều (đảo mảng, xoay mảng, đảo từ).

In [2]:
def dao_doan(a, i, j):
    while i < j:
        a[i], a[j] = a[j], a[i]
        i += 1; j -= 1

arr_tp = [1, 2, 3, 4, 5, 6]
dao_doan(arr_tp, 1, 4)
print("đảo đoạn [1..4]:", arr_tp)

đảo đoạn [1..4]: [1, 5, 4, 3, 2, 6]


### Lỗi hay gặp

- Quên **sắp xếp** trước khi dùng hai đầu cho bài tổng.
- Điều kiện `L <= R` thay vì `L < R` → dùng một phần tử hai lần.
- Không chứng minh được vì sao bỏ một đầu là an toàn → dễ bỏ mất đáp án.
- Con trỏ ghi vượt con trỏ đọc (không thể xảy ra nếu `write` chỉ tăng khi ghi).

## 2.2 Thuật toán tham lam (greedy)

### Tham lam là gì?

Thuật toán tham lam xây lời giải bằng cách ở **mỗi bước chọn phương án tốt nhất trước mắt**, không bao giờ quay lại sửa. Nhanh và đơn giản — nhưng **không phải lúc nào cũng đúng**.

### Ví dụ đúng: đổi tiền với mệnh giá 1, 2, 5, 10

Luôn lấy tờ **lớn nhất** không vượt số tiền còn lại.

### Ví dụ sai: mệnh giá 1, 3, 4, cần 6

Tham lam: `4 + 1 + 1` (3 tờ). Tối ưu: `3 + 3` (2 tờ). Chọn tốt nhất trước mắt dẫn tới kết quả tệ hơn.

In [3]:
def doi_tien_tham_lam(menh_gia, so_tien):
    chon = []
    for m in sorted(menh_gia, reverse=True):
        while so_tien >= m:
            so_tien -= m
            chon.append(m)
    return chon

def doi_tien_toi_uu(menh_gia, so_tien):           # DP để đối chiếu
    INF = float("inf")
    dp = [0] + [INF] * so_tien
    for x in range(1, so_tien + 1):
        for m in menh_gia:
            if m <= x:
                dp[x] = min(dp[x], dp[x - m] + 1)
    return dp[so_tien]

print("mệnh giá [1,2,5,10], 18:", doi_tien_tham_lam([1, 2, 5, 10], 18), "| tối ưu:", doi_tien_toi_uu([1, 2, 5, 10], 18), "tờ")
print("mệnh giá [1,3,4], 6:", doi_tien_tham_lam([1, 3, 4], 6), "| tối ưu:", doi_tien_toi_uu([1, 3, 4], 6), "tờ")

mệnh giá [1,2,5,10], 18: [10, 5, 2, 1] | tối ưu: 4 tờ
mệnh giá [1,3,4], 6: [4, 1, 1] | tối ưu: 2 tờ


### Làm sao biết tham lam đúng?

Phải **chứng minh**. Hai kỹ thuật hay dùng:

1. **Lập luận trao đổi (exchange argument)**: giả sử có lời giải tối ưu khác lựa chọn tham lam ở bước đầu; chỉ ra có thể **đổi** nó sang lựa chọn tham lam mà không tệ đi.
2. **Bất biến "luôn dẫn đầu"**: sau mỗi bước, lời giải tham lam không kém bất kỳ lời giải nào khác.

Và luôn **thử phản ví dụ** bằng brute force trên dữ liệu nhỏ.

### Ví dụ kinh điển: chọn nhiều cuộc họp nhất không trùng giờ

Chọn cuộc họp **kết thúc sớm nhất** trước — để lại nhiều thời gian nhất cho phần còn lại (lập luận trao đổi: thay cuộc họp đầu của lời giải tối ưu bằng cuộc kết thúc sớm nhất vẫn không chồng lấn).

In [4]:
def nhieu_cuoc_hop(hop):
    chon, ket_thuc = [], float("-inf")
    for s, e in sorted(hop, key=lambda x: x[1]):     # theo giờ kết thúc
        if s >= ket_thuc:
            chon.append((s, e))
            ket_thuc = e
    return chon

hop_demo = [(1, 4), (3, 5), (0, 6), (5, 7), (3, 9), (5, 9), (6, 10), (8, 11), (8, 12), (2, 14), (12, 16)]
def theo_bat_dau(hop):                               # tiêu chí SAI để so sánh
    chon, ket_thuc = [], float("-inf")
    for s, e in sorted(hop):
        if s >= ket_thuc:
            chon.append((s, e)); ket_thuc = e
    return chon

print("kết thúc sớm nhất trước:", nhieu_cuoc_hop(hop_demo), "→", len(nhieu_cuoc_hop(hop_demo)), "cuộc")
print("bắt đầu sớm nhất trước (sai):", theo_bat_dau(hop_demo), "→", len(theo_bat_dau(hop_demo)), "cuộc")

kết thúc sớm nhất trước: [(1, 4), (5, 7), (8, 11), (12, 16)] → 4 cuộc
bắt đầu sớm nhất trước (sai): [(0, 6), (6, 10), (12, 16)] → 3 cuộc


### Lỗi hay gặp

- Tin tham lam đúng mà không chứng minh / không thử phản ví dụ.
- Chọn sai tiêu chí tham lam (bắt đầu sớm nhất, ngắn nhất… thay vì kết thúc sớm nhất).
- Quên sắp xếp theo tiêu chí trước khi duyệt.

## 2.3 Áp dụng vào bài này

Dùng **hai con trỏ ở hai đầu**. Diện tích bị chặn bởi cột **thấp hơn**, nên mỗi bước ta dời cột thấp vào trong — đây là một lập luận **tham lam** có chứng minh: giữ cột thấp và dời cột kia chỉ làm chiều rộng giảm mà chiều cao không tăng, nên cặp với cột thấp không thể tốt hơn nữa và có thể bỏ.

# 3. Công thức diện tích

Với hai cột `i < j`:

```text
chiều rộng = j - i
mực nước   = min(height[i], height[j])     # nước tràn qua cột thấp hơn
diện tích  = (j - i) * min(height[i], height[j])
```

Nước **không quan tâm** tới các cột ở giữa — đề chỉ xét hai cột được chọn.

In [5]:
from typing import List

def area(height: List[int], i: int, j: int) -> int:
    return (j - i) * min(height[i], height[j])

h = [1, 8, 6, 2, 5, 4, 8, 3, 7]
print(area(h, 1, 8))   # 7 * min(8, 7) = 49
print(area(h, 1, 6))   # 5 * min(8, 8) = 40
print(area(h, 0, 8))   # 8 * min(1, 7) = 8

49
40
8


Hai yếu tố kéo co nhau:

- Cột **xa nhau** → rộng, nhưng có thể thấp.
- Cột **cao** → nhưng có thể gần nhau.

Cặp `(0, 8)` rộng nhất nhưng chỉ được `8` vì cột 0 quá thấp.

# 4. Cách dễ nghĩ nhất: thử mọi cặp

In [6]:
def max_area_bruteforce(height: List[int]) -> int:
    n = len(height)
    best = 0
    for i in range(n):
        for j in range(i + 1, n):
            best = max(best, area(height, i, j))
    return best

print(max_area_bruteforce([1, 8, 6, 2, 5, 4, 8, 3, 7]))
print(max_area_bruteforce([1, 1]))

49
1


## Độ phức tạp

Có `n(n-1)/2` cặp → `O(n²)`. Với `n = 10^5` là khoảng `5 * 10^9` cặp — không thể chạy kịp.

Câu hỏi: có cách nào **loại bỏ** hàng loạt cặp mà chắc chắn chúng không phải đáp án?

# 5. Ý tưởng: bắt đầu từ bể rộng nhất

Đặt `left = 0`, `right = n - 1` — đây là bể **rộng nhất** có thể.

Mỗi bước, ta **dời một con trỏ vào trong** (chiều rộng giảm 1). Dời con trỏ nào?

Giả sử `height[left] < height[right]`. Xét mọi bể dùng cột `left` với một cột `k` bên trong (`left < k < right`):

- Chiều rộng `k - left` **nhỏ hơn** `right - left`.
- Mực nước `min(height[left], height[k]) <= height[left]` — **không thể cao hơn** cột `left`.

→ Mọi bể như vậy đều **không lớn hơn** bể `(left, right)` hiện tại, mà bể này ta đã tính rồi.

> Vậy cột `left` (cột thấp hơn) **đã dùng hết tiềm năng**. Bỏ nó đi: `left += 1`.

Ngược lại, nếu `height[right]` thấp hơn, ta bỏ cột `right`: `right -= 1`.

Nếu hai cột bằng nhau, dời bên nào cũng được (lập luận trên áp dụng cho cả hai).

## Ví dụ minh họa lập luận

`height = [1, 8, 6, 2, 5, 4, 8, 3, 7]`, lúc đầu `left = 0` (cao 1), `right = 8` (cao 7).

Diện tích `(0, 8)` là `8 * 1 = 8`. Mọi bể khác dùng cột 0 đều có mực nước ≤ 1 và chiều rộng < 8, nên ≤ 8. Không cần thử chúng:

In [7]:
h = [1, 8, 6, 2, 5, 4, 8, 3, 7]
print("Bể (0, 8):", area(h, 0, 8))
print("Các bể (0, k) với k bên trong:", [area(h, 0, k) for k in range(1, 8)])

Bể (0, 8): 8
Các bể (0, k) với k bên trong: [1, 2, 3, 4, 5, 6, 7]


# 6. Lời giải chính

```text
left, right = 0, n - 1
while left < right:
    cập nhật best với area(left, right)
    if height[left] < height[right]: left += 1
    else: right -= 1
```

Mỗi bước loại bỏ **một cột** khỏi mọi xem xét sau này. Có `n` cột → tối đa `n - 1` bước.

In [8]:
def max_area_two_pointers(height: List[int]) -> int:
    left, right = 0, len(height) - 1
    best = 0

    while left < right:
        width = right - left
        water = min(height[left], height[right])
        best = max(best, width * water)

        if height[left] < height[right]:
            left += 1
        else:
            right -= 1

    return best

print(max_area_two_pointers([1, 8, 6, 2, 5, 4, 8, 3, 7]))
print(max_area_two_pointers([1, 1]))

49
1


# 7. Mô phỏng từng bước

`height = [1, 8, 6, 2, 5, 4, 8, 3, 7]`

| Bước | left (cao) | right (cao) | rộng | mực nước | diện tích | best | Dời |
|---|---|---|---|---|---|---|---|
| 1 | 0 (1) | 8 (7) | 8 | 1 | 8 | 8 | left (1 < 7) |
| 2 | 1 (8) | 8 (7) | 7 | 7 | **49** | 49 | right (7 < 8) |
| 3 | 1 (8) | 7 (3) | 6 | 3 | 18 | 49 | right |
| 4 | 1 (8) | 6 (8) | 5 | 8 | 40 | 49 | right (bằng nhau) |
| 5 | 1 (8) | 5 (4) | 4 | 4 | 16 | 49 | right |
| 6 | 1 (8) | 4 (5) | 3 | 5 | 15 | 49 | right |
| 7 | 1 (8) | 3 (2) | 2 | 2 | 4 | 49 | right |
| 8 | 1 (8) | 2 (6) | 1 | 6 | 6 | 49 | right |

Sau bước 8, `right = 1 = left` → dừng. Chỉ **8 bước** thay vì 36 cặp.

In [9]:
def trace_max_area(height: List[int]) -> int:
    left, right = 0, len(height) - 1
    best = 0
    step = 1
    while left < right:
        a = (right - left) * min(height[left], height[right])
        best = max(best, a)
        if height[left] < height[right]:
            move = f"dời left (cột {left} thấp hơn)"
        else:
            move = f"dời right (cột {right} thấp hơn hoặc bằng)"
        print(f"bước {step}: left={left}(h={height[left]}) right={right}(h={height[right]}) "
              f"diện tích={a:>3} best={best:>3} -> {move}")
        if height[left] < height[right]:
            left += 1
        else:
            right -= 1
        step += 1
    return best

print("Đáp án:", trace_max_area([1, 8, 6, 2, 5, 4, 8, 3, 7]))

bước 1: left=0(h=1) right=8(h=7) diện tích=  8 best=  8 -> dời left (cột 0 thấp hơn)
bước 2: left=1(h=8) right=8(h=7) diện tích= 49 best= 49 -> dời right (cột 8 thấp hơn hoặc bằng)
bước 3: left=1(h=8) right=7(h=3) diện tích= 18 best= 49 -> dời right (cột 7 thấp hơn hoặc bằng)
bước 4: left=1(h=8) right=6(h=8) diện tích= 40 best= 49 -> dời right (cột 6 thấp hơn hoặc bằng)
bước 5: left=1(h=8) right=5(h=4) diện tích= 16 best= 49 -> dời right (cột 5 thấp hơn hoặc bằng)
bước 6: left=1(h=8) right=4(h=5) diện tích= 15 best= 49 -> dời right (cột 4 thấp hơn hoặc bằng)
bước 7: left=1(h=8) right=3(h=2) diện tích=  4 best= 49 -> dời right (cột 3 thấp hơn hoặc bằng)
bước 8: left=1(h=8) right=2(h=6) diện tích=  6 best= 49 -> dời right (cột 2 thấp hơn hoặc bằng)
Đáp án: 49


# 8. Phiên bản đúng format LeetCode

In [10]:
class Solution:
    def maxArea(self, height: List[int]) -> int:
        left, right = 0, len(height) - 1
        best = 0
        while left < right:
            best = max(best, (right - left) * min(height[left], height[right]))
            if height[left] < height[right]:
                left += 1
            else:
                right -= 1
        return best

In [11]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<32} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.maxArea([1, 8, 6, 2, 5, 4, 8, 3, 7]), 49)
check("Ví dụ 2", sol.maxArea([1, 1]), 1)
check("Có cột cao 0", sol.maxArea([0, 0]), 0)
check("Tăng dần", sol.maxArea([1, 2, 3, 4, 5]), 6)
check("Giảm dần", sol.maxArea([5, 4, 3, 2, 1]), 6)
check("Cột cao ở giữa không giúp", sol.maxArea([1, 100, 100, 1]), 100)
check("Hai cột cao ở hai đầu", sol.maxArea([10, 1, 1, 1, 10]), 40)

Ví dụ 1                          PASS  49
Ví dụ 2                          PASS  1
Có cột cao 0                     PASS  0
Tăng dần                         PASS  6
Giảm dần                         PASS  6
Cột cao ở giữa không giúp        PASS  100
Hai cột cao ở hai đầu            PASS  40


## Đối chiếu với Brute Force

In [12]:
import random

random.seed(11)
for _ in range(3000):
    h = [random.randint(0, 20) for _ in range(random.randint(2, 15))]
    assert sol.maxArea(h) == max_area_bruteforce(h), h
print("3000 test ngẫu nhiên: Two Pointers khớp Brute Force")

3000 test ngẫu nhiên: Two Pointers khớp Brute Force


# 9. Vì sao thuật toán đúng? (chứng minh)

Gọi `(i*, j*)` là một cặp tối ưu. Ta chứng minh thuật toán **chắc chắn tính tới** cặp này (hoặc một cặp tốt bằng).

Lúc đầu `left = 0 <= i*` và `right = n - 1 >= j*`. Hai con trỏ chỉ đi vào trong, nên sẽ có một con trỏ chạm vị trí tối ưu của nó **trước**. Giả sử đó là `left`: có thời điểm `left = i*` và `right = r >= j*`.

Ta chứng minh `left` **không bị dời khỏi `i*`** cho tới khi `right` xuống tới `j*`:

- Giả sử ngược lại: `left` bị dời khi `right = r' > j*`. Theo quy tắc, điều đó chỉ xảy ra khi `height[i*] < height[r']`.
- Khi đó mực nước của cặp `(i*, r')` là `height[i*]`, nên:

```text
area(i*, r') = (r' - i*) * height[i*]
             > (j* - i*) * height[i*]
             >= (j* - i*) * min(height[i*], height[j*]) = area(i*, j*)
```

- Tức cặp `(i*, r')` **tốt hơn** cặp tối ưu — vô lý.

Vậy `right` sẽ đi xuống tới `j*` trong khi `left` vẫn ở `i*`, và thuật toán tính đúng cặp `(i*, j*)`. Trường hợp `right` chạm `j*` trước là đối xứng.

**Trực giác gọn hơn:** mỗi lần bỏ cột thấp hơn, ta chỉ bỏ những bể **không thể tốt hơn** bể vừa tính (mục 5).

# 10. Độ phức tạp

```text
Time:  O(n)   — mỗi bước loại một cột, tối đa n - 1 bước
Space: O(1)
```

| Cách làm | Thời gian | Bộ nhớ |
|---|---|---|
| Brute Force | O(n²) | O(1) |
| Two Pointers | O(n) | O(1) |

# 11. Những lỗi dễ mắc

## Lỗi 1: Dời cột cao hơn

Dời cột cao hơn có thể bỏ lỡ đáp án, vì cột thấp vẫn "giữ" mực nước xuống thấp.

In [13]:
def wrong_move_taller(height: List[int]) -> int:
    left, right = 0, len(height) - 1
    best = 0
    while left < right:
        best = max(best, (right - left) * min(height[left], height[right]))
        if height[left] > height[right]:     # SAI: dời cột CAO hơn
            left += 1
        else:
            right -= 1
    return best

h = [1, 8, 6, 2, 5, 4, 8, 3, 7]
print("Sai :", wrong_move_taller(h))
print("Đúng:", max_area_two_pointers(h))

Sai : 8
Đúng: 49


## Lỗi 2: Dùng `max` thay vì `min` cho mực nước

Nước tràn qua cột **thấp** hơn. `max(...)` cho diện tích quá lớn, sai.

## Lỗi 3: Tính chiều rộng sai

Chiều rộng là `right - left`, không phải `right - left + 1` (đây là khoảng cách giữa hai cột, không phải số cột).

## Lỗi 4: Nhầm với bài Trapping Rain Water (#42)

Bài #42 tính nước đọng **giữa mọi cột** (các cột giữa có tác dụng). Bài #11 chỉ chọn **hai cột**, cột giữa bị bỏ qua.

# 12. Mẹo nhận diện pattern

Nghĩ tới **Two Pointers từ hai đầu** khi:

- Cần chọn **một cặp** `(i, j)` tối ưu hóa một đại lượng phụ thuộc `j - i` và giá trị hai đầu.
- Mảng đã **sắp xếp** và cần tìm cặp có tổng/hiệu cho trước (**#167 Two Sum II**, **#15 3Sum**).
- Có thể **chứng minh**: ở mỗi bước, một trong hai đầu chắc chắn không còn dùng được.

Câu hỏi then chốt cần tự hỏi: *"Nếu dời con trỏ này, mình bỏ đi những cặp nào? Có chắc chúng không tốt hơn không?"*

## Template

```python
left, right = 0, n - 1
while left < right:
    xử lý cặp (left, right)
    if <đầu trái hết tiềm năng>:
        left += 1
    else:
        right -= 1
```

# 13. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Không chạy code. Lập bảng trace cho `height = [2, 3, 4, 5, 18, 17, 6]`. Đáp án là bao nhiêu, ở cặp nào?

## Bài 2 — Đoán output

```python
print(area([4, 3, 2, 1, 4], 0, 4))
print(area([1, 2, 1], 0, 2))
print(max_area_two_pointers([1, 2, 4, 3]))
```

## Bài 3 — Điền chỗ trống

In [14]:
def exercise_3_max_area(height: List[int]) -> int:
    left, right = 0, len(height) - 1
    best = 0
    # TODO 1: lặp khi left < right
    # TODO 2: cập nhật best với diện tích hiện tại
    # TODO 3: dời con trỏ ở cột THẤP hơn
    return best


# assert exercise_3_max_area([1, 8, 6, 2, 5, 4, 8, 3, 7]) == 49
# assert exercise_3_max_area([1, 1]) == 1
# assert exercise_3_max_area([4, 3, 2, 1, 4]) == 16

## Bài 4 — Biến thể: trả về cặp chỉ số

Trả về cặp `(i, j)` cho diện tích lớn nhất (nếu có nhiều cặp, trả về cặp **gặp đầu tiên** trong quá trình duyệt hai con trỏ).

```python
best_pair([1, 8, 6, 2, 5, 4, 8, 3, 7]) -> (1, 8)
best_pair([1, 1])                     -> (0, 1)
```

In [15]:
def exercise_4_best_pair(height: List[int]):
    # TODO
    pass


# assert exercise_4_best_pair([1, 8, 6, 2, 5, 4, 8, 3, 7]) == (1, 8)
# assert exercise_4_best_pair([1, 1]) == (0, 1)

## Bài 5 — Biến thể: Two Sum II (LeetCode #167)

Cho mảng `numbers` đã **sắp xếp tăng dần** và `target`. Tìm hai chỉ số (đánh số **từ 1**) `[i, j]`, `i < j`, sao cho `numbers[i] + numbers[j] == target`. Đề đảm bảo có đúng một đáp án. Yêu cầu bộ nhớ `O(1)`.

```python
two_sum_sorted([2, 7, 11, 15], 9) -> [1, 2]
two_sum_sorted([2, 3, 4], 6)      -> [1, 3]
two_sum_sorted([-1, 0], -1)       -> [1, 2]
```

In [16]:
def exercise_5_two_sum_sorted(numbers: List[int], target: int) -> List[int]:
    # TODO
    pass


# assert exercise_5_two_sum_sorted([2, 7, 11, 15], 9) == [1, 2]
# assert exercise_5_two_sum_sorted([2, 3, 4], 6) == [1, 3]
# assert exercise_5_two_sum_sorted([-1, 0], -1) == [1, 2]

## Bài 6 — Mini challenge: tối ưu nhỏ

Khi dời `left`, nếu cột mới **không cao hơn** cột cũ thì bể mới chắc chắn không tốt hơn (rộng hơn không được, cao hơn cũng không). Hãy viết bản bỏ qua liền một lúc mọi cột như vậy (dùng `while` bên trong), và tương tự cho `right`. Kết quả phải giống lời giải chính.

In [17]:
def exercise_6_max_area_skip(height: List[int]) -> int:
    # TODO
    pass


# assert exercise_6_max_area_skip([1, 8, 6, 2, 5, 4, 8, 3, 7]) == 49
# assert exercise_6_max_area_skip([1, 1]) == 1
# assert exercise_6_max_area_skip([2, 2, 2, 2]) == 6

# 14. Gợi ý

**Bài 1**
- Mức 1: bắt đầu `left = 0` (2), `right = 6` (6).
- Mức 2: cột 0, 1, 2, 3 lần lượt bị dời vì thấp hơn.
- Mức 3: hãy để ý cặp `(4, 5)` (cao 18 và 17) và cặp `(1, 6)`.

**Bài 3**
- Mức 1: giống template mục 12.
- Mức 2: diện tích `(right - left) * min(...)`.
- Mức 3: `if height[left] < height[right]: left += 1 else: right -= 1`.

**Bài 4**
- Mức 1: lưu thêm `best_i, best_j`.
- Mức 2: chỉ cập nhật khi diện tích **lớn hơn hẳn** `best` (dùng `>`, không `>=`).

**Bài 5**
- Mức 1: tổng nhỏ hơn `target` → cần số lớn hơn → dời `left`.
- Mức 2: tổng lớn hơn → dời `right`.
- Mức 3: nhớ cộng 1 vào chỉ số khi trả về.

**Bài 6**
- Mức 1: lưu `h = min(height[left], height[right])` trước khi dời.
- Mức 2: `while left < right and height[left] <= h: left += 1`.
- Mức 3: tương tự `while left < right and height[right] <= h: right -= 1`.

# 15. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [18]:
trace_max_area([2, 3, 4, 5, 18, 17, 6])

bước 1: left=0(h=2) right=6(h=6) diện tích= 12 best= 12 -> dời left (cột 0 thấp hơn)
bước 2: left=1(h=3) right=6(h=6) diện tích= 15 best= 15 -> dời left (cột 1 thấp hơn)
bước 3: left=2(h=4) right=6(h=6) diện tích= 16 best= 16 -> dời left (cột 2 thấp hơn)
bước 4: left=3(h=5) right=6(h=6) diện tích= 15 best= 16 -> dời left (cột 3 thấp hơn)
bước 5: left=4(h=18) right=6(h=6) diện tích= 12 best= 16 -> dời right (cột 6 thấp hơn hoặc bằng)
bước 6: left=4(h=18) right=5(h=17) diện tích= 17 best= 17 -> dời right (cột 5 thấp hơn hoặc bằng)


17

Đáp án `17`, đạt ở cặp `(4, 5)` (rộng 1 × cao 17) — lớn hơn `(2, 6)` (rộng 4 × cao 4 = 16), `(1, 6)` (rộng 5 × cao 3 = 15) hay `(3, 6)` (rộng 3 × cao 5 = 15). Ví dụ này cho thấy hai cột **sát nhau nhưng rất cao** có thể thắng.

## Đáp án Bài 2

In [19]:
print(area([4, 3, 2, 1, 4], 0, 4))            # 4 * 4 = 16
print(area([1, 2, 1], 0, 2))                  # 2 * 1 = 2
print(max_area_two_pointers([1, 2, 4, 3]))    # 4: cặp (1, 3) -> rộng 2 * cao min(2, 3) = 4

16
2
4


## Đáp án Bài 3

In [20]:
def answer_3_max_area(height: List[int]) -> int:
    left, right = 0, len(height) - 1
    best = 0
    while left < right:
        best = max(best, (right - left) * min(height[left], height[right]))
        if height[left] < height[right]:
            left += 1
        else:
            right -= 1
    return best

assert answer_3_max_area([1, 8, 6, 2, 5, 4, 8, 3, 7]) == 49
assert answer_3_max_area([1, 1]) == 1
assert answer_3_max_area([4, 3, 2, 1, 4]) == 16
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [21]:
def answer_4_best_pair(height: List[int]):
    left, right = 0, len(height) - 1
    best, best_pair = -1, None
    while left < right:
        a = (right - left) * min(height[left], height[right])
        if a > best:
            best, best_pair = a, (left, right)
        if height[left] < height[right]:
            left += 1
        else:
            right -= 1
    return best_pair

assert answer_4_best_pair([1, 8, 6, 2, 5, 4, 8, 3, 7]) == (1, 8)
assert answer_4_best_pair([1, 1]) == (0, 1)
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [22]:
def answer_5_two_sum_sorted(numbers: List[int], target: int) -> List[int]:
    left, right = 0, len(numbers) - 1
    while left < right:
        total = numbers[left] + numbers[right]
        if total == target:
            return [left + 1, right + 1]
        if total < target:
            left += 1
        else:
            right -= 1
    return []

assert answer_5_two_sum_sorted([2, 7, 11, 15], 9) == [1, 2]
assert answer_5_two_sum_sorted([2, 3, 4], 6) == [1, 3]
assert answer_5_two_sum_sorted([-1, 0], -1) == [1, 2]
print("Bài 5 PASS")

Bài 5 PASS


## Đáp án Bài 6

In [23]:
def answer_6_max_area_skip(height: List[int]) -> int:
    left, right = 0, len(height) - 1
    best = 0
    while left < right:
        h = min(height[left], height[right])
        best = max(best, (right - left) * h)
        while left < right and height[left] <= h:
            left += 1
        while left < right and height[right] <= h:
            right -= 1
    return best

assert answer_6_max_area_skip([1, 8, 6, 2, 5, 4, 8, 3, 7]) == 49
assert answer_6_max_area_skip([1, 1]) == 1
assert answer_6_max_area_skip([2, 2, 2, 2]) == 6
for _ in range(3000):
    h = [random.randint(0, 20) for _ in range(random.randint(2, 15))]
    assert answer_6_max_area_skip(h) == max_area_bruteforce(h), h
print("Bài 6 PASS (kể cả 3000 test ngẫu nhiên)")

Bài 6 PASS (kể cả 3000 test ngẫu nhiên)


# 16. Tổng kết

> Bắt đầu từ bể rộng nhất. Mỗi bước bỏ **cột thấp hơn**, vì mọi bể khác dùng cột đó đều hẹp hơn và không cao hơn — không thể tốt hơn bể vừa tính.

---

## Checklist trước khi nộp

- [ ] Mực nước dùng `min`, không dùng `max`?
- [ ] Chiều rộng là `right - left`?
- [ ] Dời con trỏ ở cột **thấp** hơn?
- [ ] Vòng lặp `while left < right`?
- [ ] Đã thử `[1, 1]`, mảng tăng dần, giảm dần, có cột cao 0?